In [27]:
import pandas as pd
pd.__version__

'3.0.3'

# Pandas DataFrame Memory Layout

## With NumPy backed data types

First we will take a look at the pandas dataframe with only NumPy ndarrays as the building blocks.

Our simple samle data only has integer and string data so we only need to be sure to use old style NumPy object strings for the string data.

In [28]:
# Store strings as NumPy objects with pandas 3.0
pd.options.future.infer_string = False

In [29]:
df = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "Name": ["Robert", "Alberto", "Piotr", "Marco"],
    "Country": ["UK", "Portugal", "Poland", "Italy"],
    "Cost": [550, 540, 200, 690],
})
df

,id,Name,Country,Cost
0,1,Robert,UK,550
1,2,Alberto,Portugal,540
2,3,Piotr,Poland,200
3,4,Marco,Italy,690


In [30]:
# Inspecting the data types we see we are using numpy int64 and numpy object
df.dtypes

id          int64
Name       object
Country    object
Cost        int64
dtype: object

## Pandas BlockManager

To have a better understanding of the pandas memory layout we can look at the internal pandas block manager.

In [31]:
mgr = df._mgr
mgr

BlockManager
Items: Index(['id', 'Name', 'Country', 'Cost'], dtype='object')
Axis 1: RangeIndex(start=0, stop=4, step=1)
NumpyBlock: slice(0, 6, 3), 2 x 4, dtype: int64
NumpyBlock: slice(1, 3, 1), 2 x 4, dtype: object

BlockManager stores two 2D block, first for integers and second for strings.

In [32]:
mgr.blocks

(NumpyBlock: slice(0, 6, 3), 2 x 4, dtype: int64,
 NumpyBlock: slice(1, 3, 1), 2 x 4, dtype: object)

We can also take a look at the values of each specific block to get the underlying numpy ndarray.

In [33]:
mgr.blocks[0].values

array([[  1,   2,   3,   4],
       [550, 540, 200, 690]])

In [34]:
mgr.blocks[1].values

array([['Robert', 'Alberto', 'Piotr', 'Marco'],
       ['UK', 'Portugal', 'Poland', 'Italy']], dtype=object)

Checking the flags we can check the numpy ndarray characteristics.

In [35]:
mgr.blocks[1].values.flags

  C_CONTIGUOUS : True
  F_CONTIGUOUS : False
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

We can also investigate that the ndarray in the case of the object dtype is holding integer numbers which are pointers to the Pyhton objects (actual strings).

In [36]:
arr = mgr.blocks[1].values
# pointers take up 8 bytes
arr.itemsize

8

And that an individual scalar of the object ndarray will me the actual Python string.

In [37]:
# Python object
str_scalar = arr[0][0]
type(str_scalar)

str

While the scalar in the integer ndarray is a numpy integer.

In [38]:
# numpy scalar
int_scalar = mgr.blocks[0].values[0][0]
int_scalar

np.int64(1)

# With PyArrow backed strings

With Pandas 3.0 string dtype became the default.
String dtype is backed by PyArrow string dat type.

Let's look at the difference in the BlockManager.

In [39]:
# Let's now store strings as PyArrow string type
pd.options.future.infer_string = True

In [40]:
df = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "Name": ["Robert", "Alberto", "Piotr", "Marco"],
    "Country": ["UK", "Portugal", "Poland", "Italy"],
    "Cost": [550, 540, 200, 690],
})
df

,id,Name,Country,Cost
0,1,Robert,UK,550
1,2,Alberto,Portugal,540
2,3,Piotr,Poland,200
3,4,Marco,Italy,690


In [41]:
df.dtypes

id         int64
Name         str
Country      str
Cost       int64
dtype: object

The object ndarray block now becomes two ExtensionBlocks, each with data from one column, backed by the PyArrow array.

In [42]:
mgr = df._mgr
mgr

BlockManager
Items: Index(['id', 'Name', 'Country', 'Cost'], dtype='str')
Axis 1: RangeIndex(start=0, stop=4, step=1)
ExtensionBlock: slice(1, 2, 1), 1 x 4, dtype: str
ExtensionBlock: slice(2, 3, 1), 1 x 4, dtype: str
NumpyBlock: slice(0, 6, 3), 2 x 4, dtype: int64

In [43]:
mgr.blocks

(ExtensionBlock: slice(1, 2, 1), 1 x 4, dtype: str,
 ExtensionBlock: slice(2, 3, 1), 1 x 4, dtype: str,
 NumpyBlock: slice(0, 6, 3), 2 x 4, dtype: int64)

In [44]:
mgr.blocks[0].values

<ArrowStringArray>
['Robert', 'Alberto', 'Piotr', 'Marco']
Length: 4, dtype: str

In [45]:
mgr.blocks[0].values._pa_array

[
  [
    "Robert",
    "Alberto",
    "Piotr",
    "Marco"
  ]
]

## In-place mutations

If we mutate one element in the pandas dataframe in case when the column is backed by pyarrow, we create a new pyarrow array!

In [46]:
df._mgr.blocks[1].values._pa_array

[
  [
    "UK",
    "Portugal",
    "Poland",
    "Italy"
  ]
]

In [47]:
df_m = df
df_m

,id,Name,Country,Cost
0,1,Robert,UK,550
1,2,Alberto,Portugal,540
2,3,Piotr,Poland,200
3,4,Marco,Italy,690


In [48]:
df_m.iloc[0,2] = 'Argentina'
df_m

,id,Name,Country,Cost
0,1,Robert,Argentina,550
1,2,Alberto,Portugal,540
2,3,Piotr,Poland,200
3,4,Marco,Italy,690


This does not only create a copy but also changes the original dataframe values as they shared the same memory!

In [49]:
df._mgr.blocks[1].values._pa_array

[
  [
    "Argentina",
    "Portugal",
    "Poland",
    "Italy"
  ]
]